<a id="top"></a>

# Demo: build the shallow agent on StateGraph

```yaml
title: "Demo: build the shallow agent on StateGraph"
keywords: langgraph, stategraph, shallow agent, node, edge, conditional edge, back-edge, toolnode, add_messages, state, chatanthropic, create_agent, behavioral equivalence, langfuse, teacher demo
estimated duration: 30
```

> **Lesson:** L14. Teacher demo -- Demo 2 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L14/demos_or_activities.md).
> Makes **live** Claude calls -- set `ANTHROPIC_API_KEY` to run the two equivalence runs (the graph
> **build** and the **diagram** run with no key). Single model: **Sonnet 4.6**, so the only variable
> vs. the L10 loop is the *shape*. Nodes call native **`ChatAnthropic`**, not the `potato_llm` seam.
> Clear outputs of live cells before committing.

## Contents

- [1. Setup -- tools, client, tracing](#1-setup----tools-client-tracing)
- [2. The typed state](#2-the-typed-state)
- [3. The two nodes](#3-the-two-nodes)
- [4. Wire it, compile it, see the graph as data](#4-wire-it-compile-it-see-the-graph-as-data)
- [5. Run for equivalence with L10 (live)](#5-run-for-equivalence-with-l07-live)
- [6. The freebies, and the prebuilt one-liner](#6-the-freebies-and-the-prebuilt-one-liner)
- [7. Takeaways](#7-takeaways)

## 1. Setup -- tools, client, tracing

Three things, all reused from earlier lessons so the only *new* thing on screen is LangGraph:

- the **shared tools** from [`common/tools.py`](../../common/tools.py) (`calculator`, `lookup`,
  `flaky_fetch`) -- the same tools L10/L11 used -- wrapped as LangChain tools so they bind into the
  graph's tool node;
- the native **`ChatAnthropic`** client (Sonnet 4.6), built only when a key is present;
- a `trace_callbacks()` helper that routes spans to the L11 Langfuse instance when configured.

**The departure, again:** from L04 on, nodes call `ChatAnthropic` directly, *not* the `potato_llm`
seam -- "frameworks bring their own client abstraction." The key still loads through
`common/config.py`.

In [1]:
from typing import Annotated, TypedDict

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.tools import StructuredTool
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common import tools as T
from fluffy_potato_curriculum.common.config import (
    get_settings,
    langfuse_configured,
    require_anthropic_key,
)

SONNET = "claude-sonnet-4-6"  # the course anchor -- ONE model, so only the graph shape is new

# Wrap the SHARED tools as LangChain tools. The functions are unchanged -- LangChain just reads
# their type hints to build an args schema, so the graph's tool node runs the exact same code.
calculator = StructuredTool.from_function(
    T.calculator, name="calculator", description="Evaluate a basic arithmetic expression."
)
lookup = StructuredTool.from_function(
    T.lookup, name="lookup", description="Look up a city's population from a small table."
)
flaky_fetch = StructuredTool.from_function(
    T.flaky_fetch, name="flaky_fetch", description="Fetch a URL. May fail in several ways."
)
LC_TOOLS = [calculator, lookup, flaky_fetch]

# Build the client only when a key is present, so a keyless restart-and-run-all still passes:
# the graph BUILD and the DIAGRAM need no key; only the run cells (section 5) do.
LIVE = bool(get_settings().anthropic_api_key)
if LIVE:
    model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=512)

# The two canonical L10/L11 tasks, reused verbatim.
CHAIN_TASK = "Use the calculator to compute 21 * 2, then tell me the population of Tokyo."
CRASH_TASK = "Fetch the URL https://crash and tell me what happened."


def trace_callbacks() -> list[object]:
    """Route spans to the shared L11 Langfuse, or [] when it isn't configured.

    With Langfuse unset the run is identical -- only the spans are absent -- so this never
    blocks a run. L11 is where the Langfuse seam is taught; here we simply reuse it.
    """
    if not langfuse_configured():
        print("Langfuse not configured -- running without tracing. (L11 wires this up.)")
        return []
    from langfuse.langchain import CallbackHandler

    return [CallbackHandler()]

[↑ Back to top](#top)

## 2. The typed state

The **state** is the object threaded through every node. For a shallow agent it is essentially the
running **message history** -- the same list L10 mutated in place -- plus one `step` counter.

Watch the reducers: `messages` uses **`add_messages`**, which *appends* (it preserves the
`tool_use`/`tool_result` pairing -- we unpack this in [L1405](L1405_lecture.ipynb)); `step` uses the
default reducer, which overwrites.

In [2]:
class AgentState(TypedDict):
    """State threaded through the agent -> tools -> agent loop.

    Example value after the chaining task finishes:
        {"messages": [HumanMessage, AIMessage(tool_use calculator), ToolMessage("42"),
                      AIMessage(tool_use lookup), ToolMessage("37000000"), AIMessage(text)],
         "step": 3}
    """

    # `messages` ACCUMULATES: the add_messages reducer appends each node's update in order.
    messages: Annotated[list[BaseMessage], add_messages]
    # `step` overwrites (default reducer) -- a tiny counter to make the typing concrete.
    step: int

[↑ Back to top](#top)

## 3. The two nodes

Two nodes, each the L10 step it replaces:

- **`agent`** -- calls the model (bound with the tool schemas) on the current messages and returns
  the response to be *appended*. This is L10's "call the model" step.
- **`tools`** -- we use LangGraph's **prebuilt `ToolNode`**: it runs every requested `tool_use` and
  appends a matching `tool_result`. This is L10's `dispatch()` step, packaged.

`handle_tool_errors=True` makes `ToolNode` turn *any* tool exception into a
`ToolMessage(status="error")` -- the L10 "convert an exception into `tool_result(is_error=True)`"
behavior. (Without it, `ToolNode` re-raises real exceptions.)

In [3]:
def agent_node(state: AgentState) -> dict[str, object]:
    """L10's 'call the model' step, as a node. Binds tools at call time so the graph
    can be BUILT without an API key (the node only runs during a live invoke)."""
    bound = model.bind_tools(LC_TOOLS)
    response = bound.invoke(state["messages"])
    return {"messages": [response], "step": state["step"] + 1}


# The prebuilt tool node -- L10's dispatch loop, supplied. opt in to error handling.
tool_node = ToolNode(LC_TOOLS, handle_tool_errors=True)


def route(state: AgentState) -> str:
    """L10's 'is there a tool_use?' branch, now a routing function on the conditional edge."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END

[↑ Back to top](#top)

## 4. Wire it, compile it, see the graph as data

The same `StateGraph` recipe from L04, plus the one new thing: the **back-edge** `tools -> agent`.
Then the payoff line -- the compiled graph can **draw itself**. That render needs **no API key**:
the control flow *is* data you can inspect.

In [4]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.set_entry_point("agent")
# the conditional edge: model emitted a tool_use -> tools, else -> END (natural termination)
builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")  # <-- THE BACK-EDGE: the only thing L14 adds to a workflow
agent_app = builder.compile()

# Control flow as DATA: the cycle agent -> tools -> agent is right there in the diagram.
print(agent_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



[↑ Back to top](#top)

## 5. Run for equivalence with L10 (live)

`invoke()` the compiled graph on the two canonical L10/L11 tasks, passing the Langfuse callback.

- **Chaining task** -> watch it issue `calculator` then `lookup` and stop -- the *same tool
  sequence* the hand-rolled loop produced.
- **`flaky_fetch` crash task** -> watch tool-failure handling work inside the `tools` node.

(The agent loop is non-deterministic -- if a run's sequence differs, that's L11's variance lesson,
not a bug. The *eval set* in [L1407](L1407_lecture.ipynb), not a single run, is how we *prove*
equivalence.)

In [5]:
def trajectory(messages: list[BaseMessage]) -> list[str]:
    """The ordered tool-call names in a message list -- the run's 'path' (L11/L12)."""
    return [c["name"] for m in messages if isinstance(m, AIMessage) for c in m.tool_calls]


if not LIVE:
    print("No ANTHROPIC_API_KEY set -- skipping the live runs. Set it to watch the agent loop.")
else:
    result = agent_app.invoke(
        {"messages": [HumanMessage(CHAIN_TASK)], "step": 0},
        config={"callbacks": trace_callbacks()},
    )
    print("tool path :", trajectory(result["messages"]))  # expect ['calculator', 'lookup']
    print("steps     :", result["step"])
    print("final text:", result["messages"][-1].text)

Langfuse not configured -- running without tracing. (L11 wires this up.)


tool path : ['calculator', 'lookup']
steps     : 2
final text: Here are the results:

1. **21 × 2 = 42**
2. **Tokyo's population** is approximately **37,000,000** (37 million people), making it one of the most populous cities in the world!


In [6]:
if LIVE:
    crash = agent_app.invoke(
        {"messages": [HumanMessage(CRASH_TASK)], "step": 0},
        config={"callbacks": trace_callbacks()},
    )
    # The tools node turned the RuntimeError into an error ToolMessage; the model recovered.
    for m in crash["messages"]:
        kind = type(m).__name__
        status = getattr(m, "status", "")
        print(f"{kind:12} {status:6} {str(m.content)[:70]!r}")

Langfuse not configured -- running without tracing. (L11 wires this up.)


HumanMessage        'Fetch the URL https://crash and tell me what happened.'
AIMessage           '[{\'text\': "I\'ll try to fetch that URL right now!", \'type\': \'text\'}, {\''
ToolMessage  error  "Error: RuntimeError('connection reset by peer')\n Please fix your mista"
AIMessage           'The fetch **failed** with a **RuntimeError: "connection reset by peer"'


[↑ Back to top](#top)

## 6. The freebies, and the prebuilt one-liner

We hand-assembled the graph so the L10 -> graph mapping was visible. Name what the framework gave
you **for free**, each against its L10 twin:

| L10 made you write... | LangGraph supplied... |
| --- | --- |
| the `while` run-driver | the compiled graph's run loop |
| the `max_steps` cap | the **recursion limit** (`GraphRecursionError`) |
| a hand-emitted trace | a built-in execution trace (L1407 + Langfuse) |
| the `dispatch()` runner | the prebuilt **`ToolNode`** (we already used it in section 3) |

And the whole-graph one-liner: `create_agent` builds *this exact graph* -- agent node, tool node,
conditional edge, back-edge -- in a single call. It's "the graph I just built, wrapped," and it is
the **ReAct** pattern, the lead-in to L15. (`langgraph.prebuilt.create_react_agent` is the older,
deprecated name.)

In [7]:
if not LIVE:
    print("No key -- skipping create_agent (it constructs a real ChatAnthropic under the hood).")
else:
    from langchain.agents import create_agent

    prebuilt = create_agent(f"anthropic:{SONNET}", LC_TOOLS)
    # Same shape as our hand-wired graph: an agent node, a tools node, a conditional edge back.
    print(prebuilt.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



[↑ Back to top](#top)

## 7. Takeaways

- **It's the same loop, drawn as a graph.** `agent -> tools -> agent` until the model stops asking
  for tools is exactly the L10 loop; only the *shape* changed.
- **The conditional edge is the L10 branch.** "Is there a `tool_use`?" moved from an `if` in Python
  to a routing function on an edge -- same decision, new location.
- **The back-edge is the agent.** `tools -> agent` is the one thing L14 adds to an L04 workflow;
  it hands the model control of the path.
- **The framework gives you the boring parts for free** -- run loop, recursion cap, trace, prebuilt
  tool node. That convenience *is* the value proposition.
- **For one loop it's break-even.** The graph earns its keep when control flow branches (L15) or
  you want the built-ins. Next: state & reducers ([L1405](L1405_lecture.ipynb)).

[↑ Back to top](#top)